In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from lucent.modelzoo import inceptionv1

device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()



We are just going to look at all the neurons before us who said cat when we said cat.  
First, I'll do the analysis for 10 neurons only, to see if we need anything else after that.  

We will look at the activation ranges of each, along with PW ranges of each (These activation ranges would be scatter plots like we did for 4e:55).   

In [ ]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from PIL import Image
from lucent.optvis import render, param, transform, objectives
import shelve
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import random

import matplotlib.pyplot as plt
import numpy as np
from lucent.modelzoo import inceptionv1
from PIL import Image
from torch.nn import functional as F
from lucent.optvis import param

from olt.act import InputOutputModelSnapshot


base_report_dir = Path("mass-train-reports")


In [ ]:
# add if needed
# from lucent.modelzoo import inceptionv1
# device = "cpu"
# model = inceptionv1(pretrained=True)
# model = model.to(device)
# model = model.eval()

# lib

In [ ]:
# from olt.act import InputOutputModelSnapshot
# acts = InputOutputModelSnapshot.get_activations(timg, model, ["mixed4d_3x3_pre_relu_conv"])
# our_act = acts["mixed4d_3x3_pre_relu_conv"]["input"]
# acts["mixed4d_3x3_pre_relu_conv"]["output"].shape, acts["mixed4d_3x3_pre_relu_conv"]["input"].shape
# from olt.tfms import transform
# current_cluster_df = main_df[
#     (main_df.layer_name == "mixed4d_3x3_pre_relu_conv") & (main_df.channel == 31) & (main_df.cluster_label == 21)
# ]
# row = current_cluster_df[current_cluster_df.imagenet_label == 93].iloc[0]
# ikey = row.input_image_key
# y, x = row.y_position, row.x_position
# pil = Image.open(f"flat-images/{ikey}.jpeg")
# timg = transform(pil)[None]
# display(pil)
# # tis just weird, and polysemantic
# from torch.nn import functional as F
# ksize = 3,3
# start_y, start_x = y-1, x-1
# end_y = start_y + ksize[0]
# end_x = start_x + ksize[1]
# print(start_y, end_y, start_x, end_x)
# thresholds = (256,)

# # try 3,4 lol
# one_patch = our_act[:, :, start_y:end_y, start_x:end_x]
# one_patch.shape
# # might need to pad in the objective for now though, since i cant capture the input, tis not very nice
# # i might have to write code to fix this, copy and keep an input capture version (Which is very important for us)
# obj = patch_across_channel_with_ksize("mixed4d_3x3_bottleneck", [one_patch], [(start_y,start_x)], ksize)
# svizs = render.render_vis(model, obj, thresholds=thresholds, show_image=False)

# plt.imshow(svizs[-1][0])
# plt.show()

# # tis just weird, and polysemantic
# from torch.nn import functional as F
# ksize = 3,3
# start_y, start_x = y-1, x-1
# end_y = start_y + ksize[0]
# end_x = start_x + ksize[1]
# print(start_y, end_y, start_x, end_x)
# thresholds = (256,)

# # try 3,4 lol
# one_patch = our_act[:, :, start_y:end_y, start_x:end_x]
# one_patch.shape
# # might need to pad in the objective for now though, since i cant capture the input, tis not very nice
# # i might have to write code to fix this, copy and keep an input capture version (Which is very important for us)
# obj = patch_across_channel_with_ksize("mixed4d_3x3_bottleneck", [one_patch], [(start_y,start_x)], ksize)
# svizs = render.render_vis(model, obj, thresholds=thresholds, show_image=False)

# plt.imshow(svizs[-1][0])
# plt.show()

# # tis just weird, and polysemantic
# from torch.nn import functional as F
# ksize = 3,3
# start_y, start_x = y-1, x-1
# end_y = start_y + ksize[0]
# end_x = start_x + ksize[1]
# print(start_y, end_y, start_x, end_x)
# thresholds = (256,)

# # try 3,4 lol
# one_patch = our_act[:, :, start_y:end_y, start_x:end_x]
# one_patch.shape
# # might need to pad in the objective for now though, since i cant capture the input, tis not very nice
# # i might have to write code to fix this, copy and keep an input capture version (Which is very important for us)
# obj = patch_across_channel_with_ksize("mixed4d_3x3_bottleneck", [one_patch], [(start_y,start_x)], ksize)
# svizs = render.render_vis(model, obj, thresholds=thresholds, show_image=False)

# plt.imshow(svizs[-1][0])
# plt.show()

In [ ]:

# TARGET_THUMBNAIL_H = 300
# TARGET_THUMBNAIL_W = 300


# def _get_uniq_imagenet_labels(current_cluster_df, n_samples):
#     uniq_imagenet_labels = current_cluster_df.imagenet_label.unique()
#     uniq_imagenet_labels = np.random.choice(
#         uniq_imagenet_labels, min(len(uniq_imagenet_labels), n_samples)
#     )
#     return uniq_imagenet_labels


# def _receptive_block(i, ksize, stride, padding, input_size=None):
#     """
#     Returns [start, end) input indices (end=exclusive) that influence
#     output position i of a conv layer.
#     """
#     start = i * stride - padding
#     end = start + ksize

#     if input_size is not None:
#         start = max(start, 0)
#         end = min(end, input_size)

#     return start, end


# def _to_int_image(arr):
#     arr = np.asarray(arr, dtype=np.float32)
#     arr -= arr.min()
#     if arr.max() > 0:
#         arr /= arr.max()
#     arr = (arr * 255).astype(np.uint8)
#     return arr


# def _get_patch(pil, model, layer_name, y, x, ksize, stride, padding):
#     timg = transform(pil)[None]
#     acts = InputOutputModelSnapshot.get_activations(timg, model, [layer_name])
#     our_act = acts[layer_name]["input"]

#     start_y, end_y = _receptive_block(y, ksize[0], stride[0], padding[0])
#     start_x, end_x = _receptive_block(x, ksize[1], stride[1], padding[1])
#     one_patch = our_act[:, :, start_y:end_y, start_x:end_x]

#     return one_patch, (start_y, end_y), (start_x, end_x)


# def _do_one_row(
#     row, model, prev_layer_name, ksize, stride, padding, thresholds=(256,)
# ):
#     ikey = row.input_image_key
#     pil = Image.open(f"flat-images/{ikey}.jpeg")

#     one_patch, (start_y, _), (start_x, _) = _get_patch(
#         pil,
#         model,
#         row.layer_name,
#         row.y_position,
#         row.x_position,
#         ksize,
#         stride,
#         padding,
#     )

#     obj = patch_across_channel_with_ksize(
#         prev_layer_name, [one_patch], [(start_y, start_x)], ksize
#     )

#     svizs = render.render_vis(model, obj, thresholds=thresholds, show_image=False)
#     return Image.fromarray(_to_int_image(svizs[-1][0])).convert("RGB")


# def _make_feature_viz(
#     df,
#     layer_name,
#     channel,
#     cluster_label,
#     model,
#     ksize,
#     stride,
#     padding,
#     prev_layer_name,
#     out_image_path,
#     n_samples=4,
# ):
#     current_cluster_df = df[
#         (df.layer_name == layer_name)
#         & (df.channel == channel)
#         & (df.cluster_label == cluster_label)
#     ]
#     uniq_imagenet_labels = _get_uniq_imagenet_labels(current_cluster_df, n_samples)
#     images = []
#     for imagenet_label in uniq_imagenet_labels:
#         row = current_cluster_df[
#             current_cluster_df.imagenet_label == imagenet_label
#         ].iloc[0]
#         im = _do_one_row(row, model, prev_layer_name, ksize, stride, padding)
#         im.thumbnail((TARGET_THUMBNAIL_W, TARGET_THUMBNAIL_H))
#         images.append(im)

#     canvas = Image.new(
#         "RGB", (TARGET_THUMBNAIL_W * len(images), TARGET_THUMBNAIL_H), "black"
#     )
#     for i, im in enumerate(images):
#         x_off = (TARGET_THUMBNAIL_W - im.width) // 2
#         y_off = (TARGET_THUMBNAIL_H - im.height) // 2
#         canvas.paste(im, (i * TARGET_THUMBNAIL_W + x_off, y_off))
#     canvas.save(out_image_path)

# import json
# from itertools import batched

# bdir = Path("feature-viz-inp-batches")
# bdir.mkdir(parents=True, exist_ok=True)

# d = {}
# for k, v in cat_parent_by_count.items():
#     if k[0] == "mixed4d_3x3_pre_relu_conv":
#         d[k] = v

# k_batches = list(batched(list(d.keys()), 16))
# for i, k_b in enumerate(k_batches):
#     content = _filtered_dict_with_keys(cat_parent_by_count, k_b)
#     with open(bdir / f"{i}.json", "w") as f:
#         json.dump(content, f)

# layer_by_params = {
#     "mixed4d_3x3_pre_relu_conv": {
#         "padding": (1,1),
#         "stride": (1,1),
#         "ksize": (3,3),
#         "prev_layer_name": "mixed4d_3x3_bottleneck",
#     },
#     "mixed4d_5x5_pre_relu_conv ": {
#         "padding": (2,2),
#         "stride": (1,1),
#         "ksize": (5,5),
#         "prev_layer_name": "mixed4d_5x5_bottleneck",
#     },
#     "mixed4d_1x1_pre_relu_conv": {
#         "padding": (0,0),
#         "stride": (1,1),
#         "ksize": (1,1),
#         "prev_layer_name": "mixed4c",
#     },
#     "mixed4d_pool_reduce_pre_relu_conv": {
#         "padding": (0,0),
#         "stride": (1,1),
#         "ksize": (1,1),
#         "prev_layer_name": "mixed4d_pool",
#     }
# }

# def make_feature_viz(
#     df,
#     layer_name,
#     channel,
#     cluster_label,
#     model,
#     out_image_path,
#     n_samples=4,
# ):
#     ps = layer_by_params[layer_name]
#     _make_feature_viz(
#         df,
#         layer_name,
#         channel,
#         cluster_label,
#         model,
#         ps["ksize"],
#         ps["stride"],
#         ps["padding"],
#         ps["prev_layer_name"],
#         out_image_path,
#         n_samples,
#     )

In [ ]:
def get_deps_df_for_single_cluster_label(df, child_layer_name, child_channel, cluster_label_in_child_neuron):
    parent_by_count = defaultdict(lambda: 0)
    child_df = df[
        (df.layer_name == child_layer_name) & 
        (df.channel == child_channel) & 
        (df.cluster_label == cluster_label_in_child_neuron)
    ]
    image_keys = child_df.input_image_key.unique()
    # df = df.set_index('input_image_key')
    # df = df.sort_index()
    for image_key in tqdm(image_keys):
        image_keys_df = df[df.input_image_key == image_key]
        child_image_keys_df = image_keys_df[image_keys_df.layer_name == child_layer_name]
        for row in child_image_keys_df.itertuples():
            y, x = row.y_position, row.x_position
            parents_image_keys_df = image_keys_df[
                (image_keys_df.layer_name != child_layer_name) &
                (image_keys_df.y_position == y) &
                (image_keys_df.x_position == x)
            ]
            for tup in parents_image_keys_df.itertuples():
                parent_key = (tup.layer_name, tup.channel, tup.cluster_label)
                parent_by_count[parent_key] += 1
    return parent_by_count

# get df

In [ ]:
csv_files = list(base_report_dir.rglob("report.csv"))
csv_files.append("mixed4e1x1_report.csv")

dfs = []
for f in tqdm(csv_files):
    df = pd.read_csv(f)
    df = df[df.cluster_label != -1].reset_index()
    dfs.append(df)

main_df = pd.concat(dfs)

main_df.head()

In [ ]:
CAT_CLUSTER_LABEL = 61
FOX_CLUSTER_LABEL = 60
# cat_parent_by_count = get_deps_df_for_single_cluster_label(
#     main_df, "mixed4e_1x1_pre_relu_conv", 55, CAT_CLUSTER_LABEL
# )

fox_parent_by_count = get_deps_df_for_single_cluster_label(
    main_df, "mixed4e_1x1_pre_relu_conv", 55, FOX_CLUSTER_LABEL
)

In [ ]:
cat_neurons_db

In [ ]:
# with shelve.open("cluster_annotations.db") as db:
#     for k, v in fox_neurons_db.items():
#         db[str(k)] = {"key": k, **v}

In [ ]:
fox_neurons_db = {}
for p in fox_parent_by_count:
    k = str(p)
    if k in cat_neurons_db:
        fox_neurons_db[k] = cat_neurons_db[k]
        # print()
    

fox_neurons_db

In [ ]:
def show_cluster(base_report_dir, layer_name, channel, cluster_label):
    report_dir = base_report_dir / layer_name / str(channel)
    jpeg = report_dir / f"cluster_{cluster_label}_combined.jpeg"
    if not jpeg.exists():
        if not report_dir.exists():
            raise Exception(f"report dir {report_dir} does not exist")
        print(f"file {jpeg} does not exist, it might be a singleton cluster")
        return
    display(Image.open(jpeg))

For each combination of layer-chan-cluster, we want:

- the count
- what that cluster signifies (hariom needs to do)
    - the simplest is to keep a dict with the same struct as parent_by_count
    - And add our comments. Im guessing we can do simple meaning and extra-comment
    - A ui would be useful here

In [ ]:
# use this to revisit, or remove older problems
import shelve
with shelve.open("cluster_annotations.db") as db:
    fox_neurons_db = dict(db)

In [ ]:
import json
with open('fox-neurons-description.json', 'w') as f:
    json.dump(fox_neurons_db, f)

In [ ]:
import json
with open('cat-neurons-description.json', 'w') as f:
    json.dump(cat_neurons_db, f)

In [ ]:
def build_input_widgets():
    """Define input widgets for a cluster. Add new widgets here as needed."""
    meaning_box = widgets.Text(
        value='',
        placeholder='Enter meaning...',
        description='Meaning:',
        layout=widgets.Layout(width='80%')
    )
    notes_box = widgets.Textarea(
        value='',
        placeholder='Enter notes...',
        description='Notes:',
        layout=widgets.Layout(width='80%', height='60px')
    )
    
    return {"meaning": meaning_box, "notes": notes_box}


def get_input_values(input_widgets):
    """Extract + clean values from the input widgets dict."""
    values = {}
    for name, w in input_widgets.items():
        val = w.value
        if isinstance(val, str):
            val = val.strip()
        values[name] = val
    return values

def get_image_filename(cluster_label, mode):
    """Map a display mode to the image filename pattern."""
    if mode == "combined":
        return f"cluster_{cluster_label}_combined.jpeg"
    elif mode == "third":
        return f"cluster_{cluster_label}_third.jpeg"
    else:
        raise ValueError(f"Unknown image mode: {mode}")


class ClusterAnnotator:
    def __init__(self, base_report_dir, parent_by_count, shelve_path="cluster_annotations.db"):
        self.base_report_dir = base_report_dir
        self.shelve_path = shelve_path
        self.keys = self._filter_unannotated(list(parent_by_count.keys()))
        self.idx = 0
        self.image_mode = "combined"

        self.input_widgets = build_input_widgets()
        self.input_box = widgets.VBox(list(self.input_widgets.values()))

        self.next_button = widgets.Button(description="Next", button_style='success')
        self.next_button.on_click(self._on_next)

        self.toggle_button = widgets.Button(description="Show: combined", button_style='info')
        self.toggle_button.on_click(self._on_toggle_image)

        self.status_label = widgets.Label(value="")
        self.button_row = widgets.HBox([self.next_button, self.toggle_button])
        self.top_box = widgets.VBox([self.status_label, self.input_box, self.button_row])
        self.image_output = widgets.Output()

        self.container = widgets.VBox([self.top_box, self.image_output])

    def _filter_unannotated(self, keys):
        with shelve.open(self.shelve_path) as db:
            existing = set(db.keys())
        return [k for k in keys if str(k) not in existing]

    def _save_current(self):
        if self.idx >= len(self.keys):
            return
        key = self.keys[self.idx]
        values = get_input_values(self.input_widgets)
        with shelve.open(self.shelve_path) as db:
            db[str(key)] = {"key": key, **values}

    def _reset_inputs(self):
        for w in self.input_widgets.values():
            w.value = ''

    def _on_next(self, b):
        self._save_current()
        self.idx += 1
        self.image_mode = "combined"
        self.toggle_button.description = "Show: combined"
        self._reset_inputs()
        self.show_current()

    def _on_toggle_image(self, b):
        self.image_mode = "third" if self.image_mode == "combined" else "combined"
        self.toggle_button.description = f"Show: {self.image_mode}"
        self.show_current()

    def show_current(self):
        with self.image_output:
            clear_output(wait=True)
            if self.idx >= len(self.keys):
                self.status_label.value = "All clusters annotated!"
                return
            layer_name, channel, cluster_label = self.keys[self.idx]
            self.status_label.value = f"[{self.idx+1}/{len(self.keys)}] {layer_name} / {channel} / {cluster_label} ({self.image_mode})"
            report_dir = self.base_report_dir / layer_name / str(channel)
            jpeg = report_dir / get_image_filename(cluster_label, self.image_mode)
            if not jpeg.exists():
                if not report_dir.exists():
                    print(f"FATAL: report dir {report_dir} does not exist")
                print(f"file {jpeg} does not exist, it might be a singleton cluster")
                return
            display(Image.open(jpeg))

    def run(self):
        display(self.container)
        self.show_current()

In [ ]:
import shelve
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image


def build_input_widgets():
    """Define input widgets for a cluster. Add new widgets here as needed."""
    meaning_box = widgets.Text(
        value='',
        placeholder='Enter meaning...',
        description='Meaning:',
        layout=widgets.Layout(width='80%')
    )
    notes_box = widgets.Textarea(
        value='',
        placeholder='Enter notes...',
        description='Notes:',
        layout=widgets.Layout(width='80%', height='60px')
    )
    
    return {"meaning": meaning_box, "notes": notes_box}


def get_input_values(input_widgets):
    """Extract + clean values from the input widgets dict."""
    values = {}
    for name, w in input_widgets.items():
        val = w.value
        if isinstance(val, str):
            val = val.strip()
        values[name] = val
    return values

def get_image_filename(cluster_label, mode):
    """Map a display mode to the image filename pattern."""
    if mode == "combined":
        return f"cluster_{cluster_label}_combined.jpeg"
    elif mode == "third":
        return f"cluster_{cluster_label}_third.jpeg"
    else:
        raise ValueError(f"Unknown image mode: {mode}")


class ClusterAnnotator:
    def __init__(self, base_report_dir, parent_by_count, shelve_path="cluster_annotations.db",
                 feature_vis_bulk_dir="feature-vis-bulk"):
        self.base_report_dir = base_report_dir
        self.shelve_path = shelve_path
        self.feature_vis_bulk_dir = Path(feature_vis_bulk_dir)
        self.keys = self._filter_unannotated(list(parent_by_count.keys()))
        self.keys = self._sort_by_feature_vis_presence(self.keys)
        self.idx = 0
        self.image_mode = "combined"

        self.input_widgets = build_input_widgets()
        self.input_box = widgets.VBox(list(self.input_widgets.values()))

        self.next_button = widgets.Button(description="Next", button_style='success')
        self.next_button.on_click(self._on_next)

        self.toggle_button = widgets.Button(description="Show: combined", button_style='info')
        self.toggle_button.on_click(self._on_toggle_image)

        self.status_label = widgets.Label(value="")
        self.button_row = widgets.HBox([self.next_button, self.toggle_button])
        self.top_box = widgets.VBox([self.status_label, self.input_box, self.button_row])
        self.image_output = widgets.Output()

        self.container = widgets.VBox([self.top_box, self.image_output])

    def _filter_unannotated(self, keys):
        with shelve.open(self.shelve_path) as db:
            existing = set(db.keys())
        return [k for k in keys if str(k) not in existing]

    def _feature_vis_path(self, key):
        layer_name, channel, cluster_label = key
        return self.feature_vis_bulk_dir / layer_name / str(channel) / str(cluster_label) / "vis.jpeg"

    def _sort_by_feature_vis_presence(self, keys):
        # Keys with an existing feature-vis-bulk image come first, preserving
        # relative order within each group (stable sort).
        return sorted(keys, key=lambda k: not self._feature_vis_path(k).exists())

    def _save_current(self):
        if self.idx >= len(self.keys):
            return
        key = self.keys[self.idx]
        values = get_input_values(self.input_widgets)
        with shelve.open(self.shelve_path) as db:
            db[str(key)] = {"key": key, **values}

    def _reset_inputs(self):
        for w in self.input_widgets.values():
            w.value = ''

    def _on_next(self, b):
        self._save_current()
        self.idx += 1
        self.image_mode = "combined"
        self.toggle_button.description = "Show: combined"
        self._reset_inputs()
        self.show_current()

    def _on_toggle_image(self, b):
        self.image_mode = "third" if self.image_mode == "combined" else "combined"
        self.toggle_button.description = f"Show: {self.image_mode}"
        self.show_current()

    def show_current(self):
        with self.image_output:
            clear_output(wait=True)
            if self.idx >= len(self.keys):
                self.status_label.value = "All clusters annotated!"
                return
            layer_name, channel, cluster_label = self.keys[self.idx]
            self.status_label.value = f"[{self.idx+1}/{len(self.keys)}] {layer_name} / {channel} / {cluster_label} ({self.image_mode})"

            feature_vis_jpeg = self._feature_vis_path(self.keys[self.idx])
            if feature_vis_jpeg.exists():
                display(Image.open(feature_vis_jpeg))

            report_dir = self.base_report_dir / layer_name / str(channel)
            jpeg = report_dir / get_image_filename(cluster_label, self.image_mode)
            if not jpeg.exists():
                if not report_dir.exists():
                    print(f"FATAL: report dir {report_dir} does not exist")
                print(f"file {jpeg} does not exist, it might be a singleton cluster")
                return
            display(Image.open(jpeg))

    def run(self):
        display(self.container)
        self.show_current()

In [ ]:
def _filtered_dict_with_keys(dic, keys):
    res = {}
    for k, v in dic.items():
        if k in keys:
            res[str(k)] = v
    return res

In [ ]:
annotator = ClusterAnnotator(base_report_dir, fox_parent_by_count)
annotator.run()

# Feature viz

In [ ]:

@objectives.wrap_objective()
def patch_across_channel_with_ksize(layer, patches, positions, ksize, batch=None):
    @objectives.handle_batch(batch)
    def inner(model):
        o = model(layer)
        mse = 0
        for patch, pos in zip(patches, positions):
            y, x = pos
            y1 = y + ksize[0]
            x1 = x + ksize[1]
            # print("yahahaha", o[:, :, y:y1, x:x1].shape, patch.shape)
            mse += F.mse_loss(o[:, :, y:y1, x:x1], patch)
        return mse
    return inner


Need to analyse this

```python
"('mixed4d_3x3_pre_relu_conv', 31, 21)": {'key': ['mixed4d_3x3_pre_relu_conv',
   31,
   21],
  'meaning': 'black-snout-like-circle',
  'notes': 'when feature viz was run on the building, we saw a bunch of those black windows\nthis makes us believe that black dot of the snout is the thing the cluster is looking for'},
```

In [ ]:
thresholds = (512,)
svizs = render.render_vis(
    model, 
    objectives.channel("mixed4d_3x3_pre_relu_conv", 31), 
    thresholds=thresholds, 
    show_image=False
)

In [ ]:
plt.imshow(svizs[-1][0])
plt.show()

In [ ]:
# now we onto the patch thing

df = pd.read_csv("mass-train-reports/mixed4d_3x3_pre_relu_conv/31/report.csv")
df.head()

In [ ]:
! ls | grep flat

In [ ]:
from olt.tfms import transform
from olt.act import InputOutputModelSnapshot

row = df[(df.imagenet_label == 685) & (df.cluster_label == 21)].iloc[0]
ikey = row.input_image_key
FLAT_BASE = Path("flat-images")

b = transform(Image.open(FLAT_BASE / f"{ikey}.jpeg"))[None]
act = InputOutputModelSnapshot.get_activations(b, model, ["mixed4d_3x3_pre_relu_conv"])["mixed4d_3x3_pre_relu_conv"]["input"]

In [ ]:
act[0, :, row.y_position, row.x_position].shape